# Krish Naik Agentic AI 3.0 Course Langchain-Middleware

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [11]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage,HumanMessage,ToolMessage
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents.middleware import PIIMiddleware
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.agents.middleware import ToolErrorMiddleware,ToolCallRequest
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.agents.middleware import LLMToolEmulator
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [4]:
@tool
def book_movie(movie:str) -> str:
    """book a movie """
    print('[tool] Inside book_movie')
    return f"the movie {movie} is booked"

@tool
def cancel_booking(movie:str) -> str:
    """cancel a movie """
    print('[tool] Inside cancel_booking')
    return f"the movie {movie} is cancelled"
    
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  print('[tool] Inside check_showtimes')
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

    

## Summarization Middleware

In [5]:
agent = create_agent(
    model='openai:gpt-5-mini',
    tools=[book_movie,cancel_booking,check_showtimes],
     middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=("tokens", 50),
            keep=("messages", 1),
        )
    ],
    )

In [7]:
result = agent.invoke({
    "messages": [
        SystemMessage(content="You are a cine bot assistant"),
        HumanMessage(content="what time does interstellar run? can u book one movie for me?")
    ]
})

Inside check_showtimes

In [8]:
from rich import print

In [9]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser wants to know the 
runtime/showtime for *Interstellar* and book one movie ticket for it.\n\n## SUMMARY\n\nThe only request so far is a
movie-ticket assistance request: the user asked, “what time does interstellar run? can u book one movie for me?” No
showtimes, theater, location, date, or booking details have been provided yet. No booking has been completed.\n\n##
ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nAsk for the missing booking details needed to check *Interstellar* showtimes 
and book a ticket, especially:\n- location/theater\n- date\n- preferred time\n- number of tickets\n- any seat 
preferences\n\nIf applicable, then check available showtimes and proceed with booking one ticket.',
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='71d9a786-dbff-446c-bb61-87b000b3b4aa'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 218,
                    'prompt_tokens': 206,
                    'total_tokens': 424,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBUPKVivy6jU10uV33RJDJHB9KKLR',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee2a-ac81-75a0-a45f-978c9aad5a68-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'call_pXjPCMHHVY5QvmhjoPBlZvOr',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 206,
                'output_tokens': 218,
                'total_tokens': 424,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        ),
        ToolMessage(
            content='7:00 PM and 10:15 PM',
            name='check_showtimes',
            id='c6f676b7-e254-4e04-868f-a577ddff8b16',
            tool_call_id='call_pXjPCMHHVY5QvmhjoPBlZvOr'
        ),
        AIMessage(
            content='I can do that. I checked showtimes and Interstellar is playing at 7:00 PM and 10:15 PM. To go 
ahead and book one ticket I need a few details from you:\n\n- Which date do you want to go?  \n- Which location or 
theater (city, ZIP code, or theater name)?  \n- Which showtime do you want (7:00 PM or 10:15 PM)?  \n- Confirm 
number of tickets — you said one, correct?  \n- Seat preference (aisle, center/middle, front, back, or no 
preference)?  \n- Payment/confirmation details: do you want me to use a saved account or do you prefer to provide 
payment info now? Also tell me the email or phone for the booking confirmation.\n\nGive me those details and I’ll 
book the ticket. If you want me to pick defaults for anything (earliest showtime, best available seat, etc.), say 
so.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 650,
                    'prompt_tokens': 390,
                    'total_

# HITL middleware

In [51]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    print('[tool] your_read_email_tool')
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    print('[tool] your_send_email_tool')
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-5.5",
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)


In [52]:
config = {'configurable':{"thread_id":"hitl"}}
result = agent.invoke({"messages" : [
    ("user", "First call the Read email aj123@gmail.com and tell me the subject. Do not call send email until read email is completed. Next, Send an email to my manager on aj1234@gmail.com, asking for a leave, dont include any reason. its for today.")
]},config = config)

your_read_email_tool

In [53]:
print(result)

{
    'messages': [
        HumanMessage(
            content='First call the Read email aj123@gmail.com and tell me the subject. Do not call send email 
until read email is completed. Next, Send an email to my manager on aj1234@gmail.com, asking for a leave, dont 
include any reason. its for today.',
            additional_kwargs={},
            response_metadata={},
            id='e642a3d6-3e09-4348-962e-8840809247a2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 115,
                    'prompt_tokens': 217,
                    'total_tokens': 332,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 85,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBUnZGUUW1fvT544h6QJY85kHMxch',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee41-9e03-70e1-bc7c-e1f653f8e54f-0',
            tool_calls=[
                {
                    'name': 'your_read_email_tool',
                    'args': {'email_id': 'aj123@gmail.com'},
                    'id': 'call_dGOY4gP5RDwfbL2lXY0pZGQr',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 217,
                'output_tokens': 115,
                'total_tokens': 332,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 85}
            }
        ),
        ToolMessage(
            content='Email content for ID: aj123@gmail.com',
            name='your_read_email_tool',
            id='41237a79-dea0-4b59-bdc8-af463719fc21',
            tool_call_id='call_dGOY4gP5RDwfbL2lXY0pZGQr'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 117,
                    'prompt_tokens': 263,
                    'total_tokens': 380,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 58,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBUncNf4RVrj029gQAPcmF7C62cHN',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee41-a810-7b20-a042-d68bec36d448-0',
            tool_calls=[
                {
                    'name': 'your_send_email_tool',
                    'args': {
                        'recipient': 'aj1234@gmail.com',
                        'subject': 'Leave Request for Today',
                        'body': 'Dear Manager,\n\nI would like to request leave for today.\n\nThank you.'
                    },
                    'id': 'call_PgTVCKdfAV2FHLGjqdDauE3k',
                    'type': 'tool_call'
           

In [54]:
resumed_result = agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

your_send_email_tool

In [55]:
print(resumed_result)

{
    'messages': [
        HumanMessage(
            content='First call the Read email aj123@gmail.com and tell me the subject. Do not call send email 
until read email is completed. Next, Send an email to my manager on aj1234@gmail.com, asking for a leave, dont 
include any reason. its for today.',
            additional_kwargs={},
            response_metadata={},
            id='e642a3d6-3e09-4348-962e-8840809247a2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 115,
                    'prompt_tokens': 217,
                    'total_tokens': 332,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 85,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBUnZGUUW1fvT544h6QJY85kHMxch',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee41-9e03-70e1-bc7c-e1f653f8e54f-0',
            tool_calls=[
                {
                    'name': 'your_read_email_tool',
                    'args': {'email_id': 'aj123@gmail.com'},
                    'id': 'call_dGOY4gP5RDwfbL2lXY0pZGQr',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 217,
                'output_tokens': 115,
                'total_tokens': 332,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 85}
            }
        ),
        ToolMessage(
            content='Email content for ID: aj123@gmail.com',
            name='your_read_email_tool',
            id='41237a79-dea0-4b59-bdc8-af463719fc21',
            tool_call_id='call_dGOY4gP5RDwfbL2lXY0pZGQr'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 117,
                    'prompt_tokens': 263,
                    'total_tokens': 380,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 58,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBUncNf4RVrj029gQAPcmF7C62cHN',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee41-a810-7b20-a042-d68bec36d448-0',
            tool_calls=[
                {
                    'name': 'your_send_email_tool',
                    'args': {
                        'recipient': 'aj1234@gmail.com',
                        'subject': 'Leave Request for Today',
                        'body': 'Dear Manager,\n\nI would like to request leave for today.\n\nThank you.'
                    },
                    'id': 'call_PgTVCKdfAV2FHLGjqdDauE3k',
                    'type': 'tool_call'
           

### Another HITL Example (interactive)

In [94]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")


@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."

@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."

@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."


@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."



cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]

def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return
    print('[decision]',decision)
    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print(resumed)
    print("Agent's final response:", resumed["messages"][-1].content)


In [95]:
config = {'configurable':{'thread_id':'hitl-demo-live-4'}}
guarded_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)


In [96]:
state = guarded_agent.get_state(config)
print(state.next)

()

In [97]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [98]:
state = guarded_agent.get_state(config)
print(state.next)

('HumanInTheLoopMiddleware.after_model',)

In [99]:
run_interactive_hitl_demo(guarded_agent, config)

The agent wants to call a guarded tool. Choose a decision:

1) approve  -- run it exactly as proposed

2) edit     -- run it, but change the booking_id first

3) reject   -- block it, with a reason sent back to the agent

4) respond  -- answer a question instead of deciding on the action

Type 1, 2, 3, or 4:  1


{'type': 'approve'}

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='5751fa1c-4d14-46c8-a6e6-014213b50171'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 282,
                    'total_tokens': 372,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBVHr8kQaJg2Htadu9PsdkcoNS53j',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fee5e-4293-7f40-8cb6-94c8af2eded3-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_HKYvGS5z2YxWDt7k4VC1NRLr',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 282,
                'output_tokens': 90,
                'total_tokens': 372,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='08fca731-040a-4671-adbf-b951240ccc53',
            tool_call_id='call_HKYvGS5z2YxWDt7k4VC1NRLr'
        ),
        AIMessage(
            content='Done — booking BK1042 has been cancelled. Do you need anything else (e.g., a refund, booking a
different time/movie, or a confirmation email)?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 319,
                    'total_tokens': 355,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBVIIbSjX3MsomdLQqw9HauqR4c3F',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019fee5e-ac18-71e0-88bd-8c01f8c3c3d0-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 319,
                'output_tokens': 36,
                'total_tokens': 355,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

Agent's final response: Done — booking BK1042 has been cancelled. Do you need anything else (e.g., a refund, 
booking a different time/movie, or a confirmation email)?

## Model Call limit middleware

In [11]:
@tool
def tell_story_movie(movie:str) -> str:
    """story of movie """
    print('[tool] Inside tell_story_movie')
    return f"the story of movie {movie} is this"

@tool
def tell_story_book(book:str) -> str:
    """story of book """
    print('[tool] Inside tell_story_book')
    return f"the story of book {book} is this"

    
call_limited_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[tell_story_movie,tell_story_book],
    checkpointer=InMemorySaver(),  # required for thread_limit to persist across calls
    middleware=[
        ModelCallLimitMiddleware( # low values taken just for example
            thread_limit=2,   # across the WHOLE conversation
            run_limit=1,       # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)
config = {'configurable':{'thread_id':'model-call-demo'}}
#result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
result = call_limited_agent.invoke({"messages": [HumanMessage('Tell me the story of movie Dunn.Then tell me the story of book best minds')]}, config=config)

Inside tell_story_movie

Inside tell_story_book

In [126]:
for i in result["messages"]:
    print(i.content, "", type(i))

Tell me the story of movie Dunn.Then tell me the story of book best minds  <class 
'langchain_core.messages.human.HumanMessage'>

<class 'langchain_core.messages.ai.AIMessage'>

the story of movie Dunn. is this  <class 'langchain_core.messages.tool.ToolMessage'>

the story of book best minds is this  <class 'langchain_core.messages.tool.ToolMessage'>

Model call limits exceeded: run limit (1/1)  <class 'langchain_core.messages.ai.AIMessage'>

In [127]:
result2 = call_limited_agent.invoke({"messages": [HumanMessage('Tell me the story of movie ddd')]}, config=config)

Inside tell_story_movie

In [128]:
for i in result2["messages"]:
    print(i.content, "", type(i))

Tell me the story of movie Dunn.Then tell me the story of book best minds  <class 
'langchain_core.messages.human.HumanMessage'>

<class 'langchain_core.messages.ai.AIMessage'>

the story of movie Dunn. is this  <class 'langchain_core.messages.tool.ToolMessage'>

the story of book best minds is this  <class 'langchain_core.messages.tool.ToolMessage'>

Model call limits exceeded: run limit (1/1)  <class 'langchain_core.messages.ai.AIMessage'>

Tell me the story of movie ddd  <class 'langchain_core.messages.human.HumanMessage'>

<class 'langchain_core.messages.ai.AIMessage'>

the story of movie ddd is this  <class 'langchain_core.messages.tool.ToolMessage'>

Model call limits exceeded: thread limit (2/2), run limit (1/1)  <class 'langchain_core.messages.ai.AIMessage'>

In [129]:
#result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
result3 = call_limited_agent.invoke({"messages": [HumanMessage('Summarize all the conversations for me')]}, config=config)

In [130]:
for i in result3["messages"]:
    print(i.content, "", type(i))

Tell me the story of movie Dunn.Then tell me the story of book best minds  <class 
'langchain_core.messages.human.HumanMessage'>

<class 'langchain_core.messages.ai.AIMessage'>

the story of movie Dunn. is this  <class 'langchain_core.messages.tool.ToolMessage'>

the story of book best minds is this  <class 'langchain_core.messages.tool.ToolMessage'>

Model call limits exceeded: run limit (1/1)  <class 'langchain_core.messages.ai.AIMessage'>

...

<class 'langchain_core.messages.ai.AIMessage'>

the story of movie ddd is this  <class 'langchain_core.messages.tool.ToolMessage'>

Model call limits exceeded: thread limit (2/2), run limit (1/1)  <class 'langchain_core.messages.ai.AIMessage'>

Summarize all the conversations for me  <class 'langchain_core.messages.human.HumanMessage'>

Model call limits exceeded: thread limit (2/2)  <class 'langchain_core.messages.ai.AIMessage'>

# Modelfallback middleware

In [138]:
try:    
    model_fallback_agent = create_agent(
        model="openai:gpt-5-mini-test",
        tools=[tell_story_movie,tell_story_book]
        
    )
    #result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
    result = model_fallback_agent.invoke({"messages": [HumanMessage('Tell me the story of movie Dunn.')]})
except:
    print('error')

error

In [139]:


try:    
    model_fallback_agent = create_agent(
        model="openai:gpt-5-mini-test",
        tools=[tell_story_movie,tell_story_book],
    middleware=[
        ModelFallbackMiddleware(
            "openai:gpt-5-mini-test2",   
            "openai:gpt-5-mini",  
        ),
    ],
    )
    #result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
    result = model_fallback_agent.invoke({"messages": [HumanMessage('Tell me the story of movie Dunn.')]})
    print(result)
except:
    print('error')

{
    'messages': [
        HumanMessage(
            content='Tell me the story of movie Dunn.',
            additional_kwargs={},
            response_metadata={},
            id='203d97d2-bcc0-468f-9f55-b217710eb4b5'
        ),
        AIMessage(
            content='Do you mean a specific film called "Dunn," or did you mean "Dune" (the novel/films)? I don\'t 
recognize a widely known movie titled exactly "Dunn." If you mean a different, lesser-known film called Dunn, 
please tell me the year, director, or a lead actor. Also tell me whether you want a full spoiler-filled plot 
summary or a short non-spoiler overview.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 413,
                    'prompt_tokens': 150,
                    'total_tokens': 563,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EBXDcKsTdWavrx9OCSTEAOCP6EdOd',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019feecf-8f65-78f0-9db3-f34859fc12f0-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 150,
                'output_tokens': 413,
                'total_tokens': 563,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 320}
            }
        )
    ]
}

# Toolcall limit middleware

In [18]:
@tool
def create_booking(movie:str) ->str:
    """create booking for a movie"""
    return f"movie {movie} booked"

@tool
def cancel_booking(movie:str) ->str:
    """cancel booking for a movie"""
    return f"movie {movie} booking cancelled"

tool_call_limit_agent = create_agent(
    model="openai:gpt-5-mini",
    checkpointer=InMemorySaver(),
    tools=[create_booking,cancel_booking],
    middleware=[
        ToolCallLimitMiddleware(run_limit=2,thread_limit=4),
        ToolCallLimitMiddleware(tool_name='cancel_booking', run_limit=1,thread_limit=1),
    ],
)
config = {"configurable": {"thread_id": "tool-limit-demo"}}
tasks = ['create','cancel','cancel','create']
from rich import print
result = None
for index,item in enumerate(tasks):
    print('------------')
    print(index,item)
    result = tool_call_limit_agent.invoke({"messages" : [HumanMessage(content=f'{item} booking for movie with id B{index}')]},config = config)
print(result)

------------

0 create

------------

1 cancel

------------

2 cancel

------------

3 create

{
    'messages': [
        HumanMessage(
            content='create booking for movie with id B0',
            additional_kwargs={},
            response_metadata={},
            id='fb603686-48ce-4111-9b37-c780a9b6bb5a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 152,
                    'total_tokens': 240,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGPXKdMA8Abw6qtrQZ9uDlNCrnEK',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ff92a-4b9b-79c1-8342-83f6c33d081a-0',
            tool_calls=[
                {
                    'name': 'create_booking',
                    'args': {'movie': 'B0'},
                    'id': 'call_RsbU5OwmlSKWC93AghwlEgFe',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 152,
                'output_tokens': 88,
                'total_tokens': 240,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='movie B0 booked',
            name='create_booking',
            id='54cb3de7-3f91-454b-8580-17c039d39a52',
            tool_call_id='call_RsbU5OwmlSKWC93AghwlEgFe'
        ),
        AIMessage(
            content='Done — your booking for movie ID B0 is confirmed.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 15,
                    'prompt_tokens': 185,
                    'total_tokens': 200,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGPYn9xW4NrWGhSXeZSTaswZUpap',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019ff92a-5359-7452-bf05-cd725b202693-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 185,
                'output_tokens': 15,
                'total_tokens': 200,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        HumanMessage(
            content='cancel booking for movie with id B1',
            additional_kwargs={},
            response_metadata={},
            id='63f4cdfb-d216-4f22-86c0-b9373712e390'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            respon

# PII Middleware

In [26]:
@tool
def create_booking(movie:str) ->str:
    """create booking for a movie"""
    return f"movie {movie} booked"

@tool
def cancel_booking(movie:str) ->str:
    """cancel booking for a movie"""
    return f"movie {movie} booking cancelled"

pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[create_booking,cancel_booking],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),    ],
)
result = pii_agent.invoke({"messages" : [HumanMessage(content='Here is my booking id: b123. My email is aj123@gmail.com. here is my credit card number 4111-1111-1111-1234. please create booking for movie dunn')]})
print(result)


{
    'messages': [
        HumanMessage(
            content='Here is my booking id: b123. My email is [REDACTED_EMAIL]. here is my credit card number 
4111-1111-1111-1234. please create booking for movie dunn',
            additional_kwargs={},
            response_metadata={},
            id='061a5ee2-c213-4662-967b-c1a73a80bbfb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 280,
                    'prompt_tokens': 189,
                    'total_tokens': 469,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGlq2MiVKbvZLY1TbVopNf49L75u',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ff93f-66dd-7652-a694-1cb966859f39-0',
            tool_calls=[
                {
                    'name': 'create_booking',
                    'args': {'movie': 'dunn'},
                    'id': 'call_XPFqaHgm3gwFHKoPCxZqsNQ2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 189,
                'output_tokens': 280,
                'total_tokens': 469,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        ToolMessage(
            content='movie dunn booked',
            name='create_booking',
            id='2776ecf8-27f1-4f30-b826-2547fff0303e',
            tool_call_id='call_XPFqaHgm3gwFHKoPCxZqsNQ2'
        ),
        AIMessage(
            content='Done — your booking for "dunn" is confirmed.\n\nI noticed you included sensitive information 
(email and credit card number). For your safety, I did not store or use that information. Please avoid sharing 
credit card numbers, full emails, passwords, or other private data in chat. If you need help with a payment or to 
update contact details, I can guide you on safe steps to do that.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 84,
                    'prompt_tokens': 222,
                    'total_tokens': 306,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGlteeFL4TelzD0LlMthy3ioOP0B',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019ff93f-7347-7121-8af8-d03e4e51810e-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 222,
                'output_tokens': 84,
                'total_tokens': 306,
                'input_token_de

### Regex check PII Middleware

In [28]:
import re

@tool
def check_booking_status(booking:str) ->str:
    """check booking status"""
    return f"{booking} is looking good."
        
def detect_booking_code(content: str) -> list[dict]:
    """Detect CineBot's own booking code format: BK followed by 4 digits."""
    matches = []
    for match in re.finditer(r"BK\d{4}", content):
        matches.append({"text": match.group(0), "start": match.start(), "end": match.end()})
    return matches
    
custom_pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[check_booking_status],
    middleware=[PIIMiddleware("booking_code", detector=detect_booking_code, strategy="mask")],
)

result = custom_pii_agent.invoke({
    "messages": [("user", "Can you check the status of my booking BK1044 for me?")]
})
print(result)

{
    'messages': [
        HumanMessage(
            content='Can you check the status of my booking ****1044 for me?',
            additional_kwargs={},
            response_metadata={},
            id='0ee43f3f-5431-4f33-b490-e4d8ae9d76aa'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 137,
                    'total_tokens': 227,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGumt8ATy7laWl8zPz4SG1jLbNGK',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ff947-de2b-7f01-8f1a-47f946023d25-0',
            tool_calls=[
                {
                    'name': 'check_booking_status',
                    'args': {'booking': '****1044'},
                    'id': 'call_O4kMLArivrcbyN4JLl0Tmb6p',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 137,
                'output_tokens': 90,
                'total_tokens': 227,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='****1044 is looking good.',
            name='check_booking_status',
            id='b4234fa9-f9ff-433f-99da-49b1366b222d',
            tool_call_id='call_O4kMLArivrcbyN4JLl0Tmb6p'
        ),
        AIMessage(
            content='I checked — booking ****1044 is looking good. No issues found.\n\nWould you like me to pull up
the full details (dates, itinerary, payment/receipt), send the confirmation, or help with changes or 
cancellation?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 182,
                    'prompt_tokens': 176,
                    'total_tokens': 358,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECGuqPa3Yr0YYNRAPUQJEANKlrFRd',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019ff947-ed27-7cc1-963a-db793817edef-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 176,
                'output_tokens': 182,
                'total_tokens': 358,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 128}
            }
        )
    ]
}

### Todo list middleware

In [9]:
@tool
def check_show_timing() ->str:
    """check show timing for all movies"""
    return f"movie dunn is at 11:00, movie fearnot is at 5:00, movie interstellar is at 9:00"

@tool
def create_booking(movie:str, tickets: int) ->str:
    """create booking for a movie"""
    return f"movie {movie} is booked for {tickets} people"
    
@tool
def cancel_booking(movie:str) ->str:
    """cancel booking for a movie"""
    return f"movie {movie} booking cancelled"

    
todo_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[check_show_timing, create_booking, cancel_booking],
    middleware=[TodoListMiddleware()]
)
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good science related, and book 2 seats.")]
})
from rich import print
print(result)

{
    'messages': [
        HumanMessage(
            content="I want to plan a movie night: check what's showing, pick something good science related, and 
book 2 seats.",
            additional_kwargs={},
            response_metadata={},
            id='427b5ac0-0c63-419b-bb44-7d6b47052540'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 718,
                    'prompt_tokens': 1375,
                    'total_tokens': 2093,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 640,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECWvO1Z6GUh5soPOqt1Q8kx5BeQkb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffcf2-ebbf-7f40-be9d-248520e2f6e1-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': 'Check show timings for all movies', 'status': 'in_progress'},
                            {'content': 'Select a science-related movie to watch', 'status': 'pending'},
                            {'content': 'Book 2 seats for chosen movie', 'status': 'pending'},
                            {'content': 'Confirm booking details with user', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_ESLsLTk996sPpOWaD9CbsdON',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1375,
                'output_tokens': 718,
                'total_tokens': 2093,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 640}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': 'Check show timings for all movies', 'status': 
'in_progress'}, {'content': 'Select a science-related movie to watch', 'status': 'pending'}, {'content': 'Book 2 
seats for chosen movie', 'status': 'pending'}, {'content': 'Confirm booking details with user', 'status': 
'pending'}]",
            name='write_todos',
            id='5f25202f-285b-468b-8326-57cbba5cb0e4',
            tool_call_id='call_ESLsLTk996sPpOWaD9CbsdON'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 15,
                    'prompt_tokens': 1534,
                    'total_tokens': 1549,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 1152
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ECWvYIsNe66hYau4kDDFhwWYOP

## LLMToolSelectorMiddleware

In [3]:
@tool
def check_show_timing() ->str:
    """check show timing for all movies"""
    return f"movie dunn is at 11:00, movie fearnot is at 5:00, movie interstellar is at 9:00"

@tool
def create_booking(movie:str, tickets: int) ->str:
    """create booking for a movie"""
    return f"movie {movie} is booked for {tickets} people. Here is booking id B456"
    
@tool
def cancel_booking(bookingid:str) ->str:
    """cancel booking for a bookingid"""
    return f" {bookingid} booking cancelled"
    
@tool
def refund_booking(bookingid:str) ->str:
    """refund booking for a bookingid"""
    return f" {bookingid} booking refunded"
    
@tool
def get_refund_policy() ->str:
    """defines refund policy for cancellation"""
    return f"Only half the price will be refunded. "
    
@tool
def confirm_booking(bookingid:str) ->str:
    """confirm booking for a bookingid"""
    return f" {bookingid} booking confirmed"
    
@tool
def confirm_cancellation(bookingid:str) ->str:
    """confirm booking for a bookingid"""
    return f" {bookingid} booking cancellation confirmed"

from langchain.agents.middleware import wrap_model_call
@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])

    return handler(request)

    
tool_selector_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[check_show_timing, create_booking, cancel_booking,confirm_booking,get_refund_policy],
    middleware=[LLMToolSelectorMiddleware( 
            model="openai:gpt-5-mini",     # can be a CHEAPER model than the main agent
            max_tools=2, #only give two tools at any given time
            always_include=["get_refund_policy"]),
               show_tools
            ]
)
result = tool_selector_agent.invoke({
    "messages": [HumanMessage(content="Can you cancel my booking with ID B1234?.A Then book again for another movie dunn")]
})
from rich import print
print(result)


TOOLS SENT TO MODEL:
['create_booking', 'cancel_booking', 'get_refund_policy']

TOOLS SENT TO MODEL:
['create_booking', 'cancel_booking', 'get_refund_policy']


{
    'messages': [
        HumanMessage(
            content='Can you cancel my booking with ID B1234?.A Then book again for another movie dunn',
            additional_kwargs={},
            response_metadata={},
            id='5a602cb1-425b-47e6-aac8-d53c93a9d1e2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 346,
                    'prompt_tokens': 186,
                    'total_tokens': 532,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ED9UdzaKmlsL0DkMTe4wbdjJrSCdX',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a005c9-2179-7702-a88d-976f93520fdb-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'bookingid': 'B1234'},
                    'id': 'call_Znn4738oZfb0TonHmAm4rDir',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 186,
                'output_tokens': 346,
                'total_tokens': 532,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 320}
            }
        ),
        ToolMessage(
            content=' B1234 booking cancelled',
            name='cancel_booking',
            id='ce93f24f-7a30-41dc-b04f-7617a44b1cb4',
            tool_call_id='call_Znn4738oZfb0TonHmAm4rDir'
        ),
        AIMessage(
            content='Done — your booking B1234 has been cancelled.\n\nI can rebook the movie "dunn" for you. How 
many tickets would you like, and any preferred date/time or seating? If you want me to proceed now with 1 ticket, 
say “book 1” (or tell me the number and any preferences).',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 332,
                    'prompt_tokens': 222,
                    'total_tokens': 554,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-ED9UjdXyrmahW13vE9dFkTj4Mrjm1',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a005c9-3a8f-7d12-bcef-7e7588a52027-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 222,
                'output_tokens': 332,
                'total_tokens': 554,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        )
    ]
}

# ToolErrorMiddleware

In [18]:
@tool
def divide_into_half(num: int) ->int:
    """this is a divide into half tool"""
    print(f'[tool call] divide_into_half {num}')
    return num/0
    


def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    print(f'on_error {request.tool_call['name']}, failed with {type(exc).__name__}. ')
    return f"`{request.tool_call['name']}` failed with {type(exc).__name__}."
    # propagate everything else


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[divide_into_half],
    middleware=[ToolErrorMiddleware(on_error)],
)
try:
    result = agent.invoke({"messages": [HumanMessage(content="Divide 5 into half")]})
    from rich import print
    print(result)
except Exception as e:
    print('error')

divide_into_half 5

on_error divide_into_half, failed with ZeroDivisionError.

{
    'messages': [
        HumanMessage(
            content='Divide 5 into half',
            additional_kwargs={},
            response_metadata={},
            id='0dfb6d0c-b03e-4041-b60d-5b855627c42a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 216,
                    'prompt_tokens': 132,
                    'total_tokens': 348,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EDAqhSJvK0LiX0j6J99OCsRHWg8pT',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a00618-a984-7983-8a53-ffc58af98f1b-0',
            tool_calls=[
                {
                    'name': 'divide_into_half',
                    'args': {'num': 5},
                    'id': 'call_XdWEq9ucYsgOHZPNHyOhpZrm',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 132,
                'output_tokens': 216,
                'total_tokens': 348,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        ),
        ToolMessage(
            content='`divide_into_half` failed with ZeroDivisionError.',
            name='divide_into_half',
            id='16d0a8be-1680-4956-ab59-79d1bdf89cae',
            tool_call_id='call_XdWEq9ucYsgOHZPNHyOhpZrm',
            status='error'
        ),
        AIMessage(
            content='Do you mean "split 5 in half" or "divide 5 by one-half"?\n\n- Split 5 in half (5 ÷ 2) = 2.5 
(each half).\n- Divide 5 by one-half (5 ÷ 0.5 = 5 × 2) = 10.\n\nWhich one did you intend?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 274,
                    'prompt_tokens': 173,
                    'total_tokens': 447,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EDAqkGZkA6k2lf3cBubufOzftGr32',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a00618-b5ce-7782-8877-3b7c4d9e0a24-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 173,
                'output_tokens': 274,
                'total_tokens': 447,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        )
    ]
}

# ToolRetryMiddleware

In [23]:
@tool
def divide_into_half(num: int) ->int:
    """this is a divide into half tool"""
    print(f'[tool call] divide_into_half {num}')
    if num == 0:
        raise ValueError('number is zero')
    return num/0
    

def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    print(f'on_error {request.tool_call['name']}, failed with {type(exc).__name__}. ')
    return "Make sure to give a non zero number" 
    # propagate everything else

print('testing out toolretry only')
agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[divide_into_half],
    middleware=[
                ToolRetryMiddleware(max_retries=5, on_failure="error")
               ],
)
try:
    result = agent.invoke({"messages": [HumanMessage(content="Divide 0 into half")]})
    from rich import print
    print(result)
except Exception as e:
    print('error')


testing out toolretry only

divide_into_half 0

divide_into_half 0

divide_into_half 0

divide_into_half 0

divide_into_half 0

divide_into_half 0

error

## Retry with error both middlewares together

In [35]:
count: int = 0;
@tool
def divide_into_half(num: int) ->int:
    """this is a divide into half tool"""
    global count
    count+=1
    print(f'[tool call] divide_into_half num = {num},  execution count = {count}')
    
    if num == 0:
        raise ValueError('number is zero')
    return num/0
    

def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    print(f'on_error {request.tool_call['name']}, failed with {type(exc).__name__}. ')
    return "Make sure to give a non zero number" 
    # propagate everything else

def on_error2(exc: Exception, request: ToolCallRequest) -> str | None:
    print(f'on_error2 {request.tool_call['name']}, failed with {type(exc).__name__}. ')
    if count <= 3:
        raise ValueError('number is zero')
    return "Make sure to give a non zero number" 
    
print('testing out toolerror and then toolretry middleware both together in that order')
agent1 = create_agent(
    model="openai:gpt-5-mini",
    tools=[divide_into_half],
    middleware=[
                ToolErrorMiddleware(on_error),
                ToolRetryMiddleware(max_retries=5, on_failure="error")
               ],
)
try:
    result = agent1.invoke({"messages": [HumanMessage(content="Divide 0 into half")]})
    from rich import print
    #print(result)
except Exception as e:
    print('error')

print('testing out toolretry and then toolerror middleware both together in that order')
count = 0

agent2 = create_agent(
    model="openai:gpt-5-mini",
    tools=[divide_into_half],
    middleware=[
                ToolRetryMiddleware(max_retries=5, backoff_factor=2.0, initial_delay=1.0, on_failure="error"),
                ToolErrorMiddleware(on_error),
               ],
)
try:
    result = agent2.invoke({"messages": [HumanMessage(content="Divide 0 into half.")]})
    from rich import print
    #print(result)
except Exception as e:
    print('error')

print('testing out toolretry and then toolerror middleware both together in that order without error handling')
count = 0

agent3 = create_agent(
    model="openai:gpt-5-mini",
    tools=[divide_into_half],
    middleware=[
                ToolRetryMiddleware(max_retries=5, backoff_factor=2.0, initial_delay=1.0, on_failure="error"),
                ToolErrorMiddleware(on_error2),
               ],
)
try:
    result = agent3.invoke({"messages": [HumanMessage(content="Divide 0 into half.")]})
    from rich import print
    #print(result)
except Exception as e:
    print('error')

testing out toolerror and then toolretry middleware both together in that order

divide_into_half num = 0,  execution count = 1

divide_into_half num = 0,  execution count = 2

divide_into_half num = 0,  execution count = 3

divide_into_half num = 0,  execution count = 4

...

on_error2 divide_into_half, failed with ValueError.

divide_into_half num = 0,  execution count = 3

on_error2 divide_into_half, failed with ValueError.

divide_into_half num = 0,  execution count = 4

on_error2 divide_into_half, failed with ValueError.

# LLMToolEmulator Middleware

In [20]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    print(f'[tool] get_weather {location}')
    return f"Weather in {location}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    print(f'[tool] send_email {to} {subject}  {body}')
    return "Email sent"


# Emulate all tools (default behavior)
emulator_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="openai:gpt-5-mini")]
)

# Emulate specific tools only
emulator_agent2 = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model="openai:gpt-5-mini", tools=["get_weather"])],
)
result = emulator_agent.invoke({"messages": [HumanMessage(content="what is the weather in tokyo?.  send an email to aj123@gmail.com with subject langchain and body here is what we learned about middleware")]})
from rich import print
print(result)
result = emulator_agent2.invoke({"messages": [HumanMessage(content="what is the weather in tokyo?.  send an email to aj123@gmail.com with subject langchain and body here is what we learned about middleware")]})
print(result)

{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo?.  send an email to aj123@gmail.com with subject langchain and 
body here is what we learned about middleware',
            additional_kwargs={},
            response_metadata={},
            id='a6b976c3-faef-432f-bb46-7790ba3f45f9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 263,
                    'prompt_tokens': 182,
                    'total_tokens': 445,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EDVqUCTgrpi7i94AM8Ej5ZF3jfU3B',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a00ae8-2fed-7be0-a3cd-38263b467c86-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_V4LJXrvXw8TJSqCrId5V71w9',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {
                        'to': 'aj123@gmail.com',
                        'subject': 'langchain',
                        'body': 'here is what we learned about middleware'
                    },
                    'id': 'call_9Oh3f5lieWCEQXAupXwz7APL',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 182,
                'output_tokens': 263,
                'total_tokens': 445,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        ),
        ToolMessage(
            content='{\n  "location": "Tokyo, Japan",\n  "coordinates": { "lat": 35.6895, "lon": 139.6917 },\n  
"timezone": "Asia/Tokyo",\n  "observed_at": "2026-08-16T09:22:00+09:00",\n  "temperature": { "C": 29.1, "F": 84.4 
},\n  "feels_like": { "C": 32.3, "F": 90.1 },\n  "humidity_pct": 68,\n  "pressure_hPa": 1012,\n  "wind": {\n    
"speed_kph": 11.3,\n    "speed_mps": 3.14,\n    "gust_kph": 18.5,\n    "deg": 45,\n    "dir": "NE"\n  },\n  
"visibility_km": 10,\n  "condition": { "main": "Clouds", "description": "broken clouds" },\n  "precipitation": { 
"last_1h_mm": 0.0, "today_mm": 0.0, "probability_next_1h_pct": 10 },\n  "uv_index": 6,\n  "sunrise": 
"2026-08-16T04:43:00+09:00",\n  "sunset": "2026-08-16T18:37:00+09:00",\n  "forecast_next_hours": [\n    { "time": 
"2026-08-16T10:00:00+09:00", "temp_C": 30.0, "condition": "partly cloudy", "precip_prob_pct": 10 },\n    { "time": 
"2026-08-16T13:00:00+09:00", "temp_C": 32.0, "condition": "mostly sunny", "precip_prob_pct": 5 },\n    { "time": 
"2026-08-16T16:00:00+09:00", "temp_C": 31.0, "condition": "scattered clouds", "precip_prob_pct": 15 }\n  ],\n  
"source": "simulated_openweather_api",\n  "units": "metric"\n}',
            name='get_weather',
            id='29bcd384-d9cc-4fe5-9a70-cff7118161d6',
            tool_call_id='call_V4LJXrvXw8TJSqCrId5V71w9'
        ),
        ToolMessage(
            content='{\n  "status": "sent",\n  "message_id": "<CANv1d2e3f4g5h6@mail.example.com>",\n  "to": 
"aj123@gmail.com",\n  "subject": "langchain",\n 

send_email aj123@gmail.com langchain  here is what we learned about middleware

{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo?.  send an email to aj123@gmail.com with subject langchain and 
body here is what we learned about middleware',
            additional_kwargs={},
            response_metadata={},
            id='26f6db1a-e985-4f02-8955-3ce903812653'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 329,
                    'prompt_tokens': 182,
                    'total_tokens': 511,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EDVr01RTeXBJjGanH8B0arWZAAAaG',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a00ae8-adb1-7532-9b93-1e5e3a43fe4f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo, Japan'},
                    'id': 'call_EVyUer3dx49VNimhgcgfDh2t',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {
                        'to': 'aj123@gmail.com',
                        'subject': 'langchain',
                        'body': 'here is what we learned about middleware'
                    },
                    'id': 'call_2OwGUtVxlgVR04RgTEt9xNeO',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 182,
                'output_tokens': 329,
                'total_tokens': 511,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        ToolMessage(
            content='{\n  "location": "Tokyo, Japan",\n  "latitude": 35.6895,\n  "longitude": 139.6917,\n  
"timezone": "Asia/Tokyo",\n  "timestamp": "2026-08-16T14:22:00+09:00",\n  "units": "metric",\n  "weather": {\n    
"summary": "Broken clouds with isolated light rain",\n    "conditions": [\n      {\n        "id": 803,\n        
"main": "Clouds",\n        "description": "broken clouds",\n        "icon": "04d"\n      },\n      {\n        "id":
500,\n        "main": "Rain",\n        "description": "light rain",\n        "icon": "10d"\n      }\n    ],\n    
"temperature_c": 30.6,\n    "feels_like_c": 34.1,\n    "dew_point_c": 24.8,\n    "humidity_percent": 72,\n    
"pressure_hpa": 1010,\n    "visibility_m": 10000,\n    "clouds_percent": 65,\n    "wind": {\n      "speed_m_s": 
3.6,\n      "deg": 310,\n      "gust_m_s": 6.8\n    },\n    "precipitation": {\n      "type": "rain",\n      
"last_1h_mm": 0.7,\n      "last_3h_mm": 1.4,\n      "probability": 0.35\n    },\n    "uv_index": 7\n  },\n  
"sunrise": "2026-08-16T04:57:00+09:00",\n  "sunset": "2026-08-16T18:31:00+09:00",\n  "alerts": [],\n  "source": 
"simulated"\n}',
            name='get_weather',
            id='5c709f71-f7be-4648-8c95-f36bd6e1bb82',
            tool_call_id='call_EVyUer3dx49VNimhgcgfDh2t'
        ),
        ToolMessage(
            content='Email sent',
            name='send_email',
            id='672f09db-fdb6-4bfc-bb12-c1c321599093',
            tool_call_id='call_2OwGUtVxlgVR04RgTEt9xNeO'
        ),
        AIM